In [59]:
import os
import pandas as pd
import numpy as np
import duckdb

start with looking at the head of all the smaller files but chartevents
generating the csvs for all the smaller files: d_items,icustays,patients and admissions

In [60]:
d_items_path='../data/d_items.csv.gz'
icustays_path='../data/icustays.csv.gz'
patients_path='../data/patients.csv.gz'
admissions_path='../data/admissions.csv.gz'
chartevents_path='../data/chartevents.csv.gz'
d_items=pd.read_csv(d_items_path)
icustays=pd.read_csv(icustays_path)
patients=pd.read_csv(patients_path)
admissions=pd.read_csv(admissions_path)
print("all the files have been converted to csvs")

all the files have been converted to csvs


In [ ]:
d_items.head()

<h5>inferences: everything except item id, label and abbrev is pretty insignificant so just the first 3 is sufficient for now and check which is most linked to preferably its chartevents</h5>



In [ ]:
patients.head()

<h5>inferences: subject id will be the only fruitful column in this table and will link to icustays.</h5>



In [ ]:
icustays.head()

<h5>as said above stay id is linked to subj id here but intime and outtime is pretty useful from here.</h5>



In [ ]:
admissions.head()

<h5>inferences: not completely sure of the distinction between admittime, intime etc check doc and almost everything but the first few cols is redundant</h5>



In [ ]:
# for chartevents alone:
chartevents_preview=pd.read_csv('../data/chartevents.csv.gz',nrows=5)

chartevents_preview.head()

<h4>most important one linking all of them, need to take batches of data based on the subject id and then cumulate all the values from the items</h4>
<h5> itemid from d_items must be subs here and grouping done by subject_id;</h5>



In [61]:
print("patients columns:",patients.columns.tolist())
print("icustays columns:",icustays.columns.tolist())
print("admissions columns:",admissions.columns.tolist())
print("d_items columns:",d_items.columns.tolist())
print("chartevents columns:",chartevents_preview.columns.tolist())

patients columns: ['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']
icustays columns: ['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los']
admissions columns: ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']
d_items columns: ['itemid', 'label', 'abbreviation', 'linksto', 'category', 'unitname', 'param_type', 'lownormalvalue', 'highnormalvalue']
chartevents columns: ['subject_id', 'hadm_id', 'stay_id', 'caregiver_id', 'charttime', 'storetime', 'itemid', 'value', 'valuenum', 'valueuom', 'warning']


In [62]:
#to create a a dictionary with most of the important vitals there are
print(d_items[d_items['label'].str.contains('heart rate', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('respiratory rate', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('systolic', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('temperature', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('diastolic', case=False, na=False)])
print("-------------------------------")
print(d_items[d_items['label'].str.contains('oxygen', case=False, na=False)])


   itemid                    label     abbreviation      linksto  \
2  220045               Heart Rate               HR  chartevents   
3  220046  Heart rate Alarm - High  HR Alarm - High  chartevents   
4  220047   Heart Rate Alarm - Low   HR Alarm - Low  chartevents   

              category unitname param_type  lownormalvalue  highnormalvalue  
2  Routine Vital Signs      bpm    Numeric             NaN              NaN  
3               Alarms      bpm    Numeric             NaN              NaN  
4               Alarms      bpm    Numeric             NaN              NaN  
-------------------------------
     itemid                           label                    abbreviation  \
28   220210                Respiratory Rate                              RR   
799  224688          Respiratory Rate (Set)          Respiratory Rate (Set)   
800  224689  Respiratory Rate (spontaneous)  Respiratory Rate (spontaneous)   
801  224690        Respiratory Rate (Total)        Respiratory Rate

In [ ]:
#mapping dictionary for the vitals from d_items file:
vitals_dict={"spO2":220277,"temp(C)":223762,"temp(F)":223761,"heartrate":220045,"ARTsys":225309,"ARTdia":225310,"ARTmean":225312,"NBPs":220179,"NBPd":220180,"NBPm":220181,"RR":220210}
vitals,ids=list(vitals_dict.keys()),list(vitals_dict.values())
vitals_rev_dict={vitals_dict[i]:i for i in vitals_dict}
print(vitals,ids,vitals_rev_dict)

update: take the temperature in farenheit also cuz it will has more data than celcius and for the missing values in temp C, substitute with tempeaarture from Farenheit part ; new update : not adding glucose

In [ ]:
con=duckdb.connect()

In [ ]:
# result["vitals"] = result["itemid"].map(vitals_rev_dict)

# result

group the chartevent work by itemids instead and then map the respective vitals

thats the count of each vital present in the data frame so heartrate is obviously to be taken 

In [ ]:
# WERE GETTING THE CHARTEVENTS DF FOR ONCE NOW TO CHECK ITS STATS AND THEN WE SHALL PIVOT IT

In [ ]:
querychev=f"""
SELECT subject_id,stay_id,charttime,itemid,valuenum
FROM read_csv_auto('{chartevents_path}')
WHERE itemid in {ids}
"""
rawchev=con.execute(querychev).fetch_df()


In [ ]:
rawchev.head()

In [ ]:
rawchev.shape

In [ ]:
rawchev["vitals"]=rawchev["itemid"].map(vitals_rev_dict)
rawchev.head()

In [ ]:
pivoted = rawchev.pivot(values="valuenum",columns="vitals",index=["subject_id","stay_id","charttime"])
#df.pivot_table(values='v', index='a', columns='b', aggfunc='mean')

In [ ]:
pivoted.head()

In [ ]:
pivoted.isna().sum()


In [ ]:
pivoted.isna().mean()*100

In [ ]:
icustays.head()

In [ ]:
icustays.columns

In [ ]:
icustays['intime']=pd.to_datetime(icustays['intime'])
icustays['outtime']=pd.to_datetime(icustays['outtime'])
icustays["stayhours"]=(icustays['outtime']-icustays['intime']).dt.total_seconds()/3600
icustays.head()
icustays["stayhours"].describe()
#take icu stays here cuz we need those entries who have a tangible amount of time spent under supervision which is atleast 4 hrs

In [ ]:
icustayswmorethan4 = icustays[icustays["stayhours"] >= 4].copy()
icustayswmorethan4.head()

In [ ]:
icustayswmorethan4.describe()

In [ ]:
stayids=icustayswmorethan4["stay_id"]
len(stayids)
# this is the list of stay ids which have more than 4 hrs of stay time so well filter jjust this form the entire dataset

In [ ]:
pivoted.columns
pivoted = pivoted.reset_index()
pivoted.columns
#always reset indices before manipulating and then set the index later. columns and indices go speraetly in terms of fn calls

In [ ]:
pivoted=pivoted[pivoted["stay_id"].isin(stayids)].copy()
pivoted.head()
#now get just the stay ids who have more than 4 hrs

In [ ]:
df=pivoted.sort_values(["stay_id","charttime"]).reset_index(drop=True)
df.head()

In [ ]:
exid=df["stay_id"].iloc[0]
examplestay=df[df["stay_id"]==exid].copy()
examplestay.head()
#looking at one stay for now , avoidable 

In [ ]:
examplestay["timegap"]=examplestay["charttime"].diff()
examplestay.head()

In [ ]:
df["time_diff"]=df.groupby("stay_id")["charttime"].diff()
df.head()

In [ ]:
df["time_diff"].describe()

In [ ]:
df.loc[df["time_diff"].nlargest(10).index,
    ["subject_id", "stay_id", "charttime", "time_diff"]
]

we see the same 2 subject ids having an exorbitant gap but is later removed.

In [ ]:
vital_cols=[
    "ARTdia", "ARTmean", "ARTsys",
    "NBPd", "NBPm", "NBPs",
    "RR", "heartrate", "spO2", "temp(C)","temp(F)"
]

In [ ]:
df_check=df.merge(icustays[["stay_id","intime","outtime"]], on="stay_id",how='left')
#pd.merge(df1, df2, on='key', how='inner') its an sql type inner join
df_check.head()

In [ ]:
timethres=pd.Timedelta(hours=60)
diffthres=pd.Timedelta(days=1)
df_timeclean=df_check[
    (df_check["charttime"]>=df_check["intime"]-timethres)
      & 
    (df_check["charttime"]<=df_check["outtime"]+timethres)
     & 
    ((df_check["time_diff"].isna()))
      | 
    (df_check["time_diff"]<=diffthres)
    ].copy()
#setup a time threshold, we need only those entries which fall well within the intime and outttime bracket and the difference must be less than the setup thresholds.

In [ ]:
uniform=(
    df_timeclean.set_index("charttime")
      .groupby("stay_id")[vital_cols]
      .resample("20min")
      .mean()
      .reset_index()
)
#resampling the dataset to 20 minute intervals as 5 mins didnt work well (NaN)

In [ ]:
uniform.head()

In [ ]:
uniform=uniform.sort_values(["stay_id", "charttime"])
uniform=uniform.set_index("charttime")

In [ ]:
uniform.head()

In [ ]:
uniform.groupby("stay_id").size().sort_values(ascending=False).head(10)

we get an abnormal reading spanning 1 year so it got split into 106830 samples of 5 minutes

In [ ]:
uniform = uniform.reset_index()

uniform["time_diff"] = (
    uniform.groupby("stay_id")["charttime"].diff()
)

uniform = uniform.set_index("charttime")

In [ ]:
uniform["time_diff"].describe()

FINALLY all of the data has been made into workable time series data of 20* mins intervals after removing the stray huge interval samples

In [ ]:

(uniform.isna().mean() * 100).sort_values(ascending=False)

In [ ]:
for i in vital_cols:
    actual = uniform[uniform[i].notna()].copy()

    actual["gap"] = (actual.index.to_series().groupby(actual["stay_id"]).diff())
    print(f"\n{i}")
    print(actual["gap"].dropna().describe())
    #find the time gap between 2 successive readings in each of these vitals. and then we can restrucutre it.

so for: spo2: avg is 57 mins, temp is 1hr 19 mins, heart rate is 56mins art sys and dia (and hence mean) is 1 hr 2 mins nbps nbpd and nbpm is 1hr 21 mins rr is 55 mins so take the avrage around 1.5 hrs

In [ ]:
uniform.shape

In [ ]:
uniform.memory_usage(deep=True).sum() / 1024**3
#space occupied by uniform ~ 2gigs

In [ ]:
for col in vital_cols:
    uniform[col] = uniform.groupby("stay_id")[col].ffill(limit=4)

In [ ]:
uniform[vital_cols].isna().sum()

In [ ]:
uniform[vital_cols].isna().sum()

In [ ]:
(uniform[vital_cols].isna().mean() * 100).sort_values(ascending=False)
#artdia and sys have extremely high amount of Nans

In [ ]:
uniform = uniform.drop(columns=["ARTdia","ARTsys"])
uniform.head()

In [ ]:
# d_items[d_items["label"].str.contains(
#     "glucose|fio2|oxygen|gcs|pain|urine|respiratory|ventilator",
#     case=False,
#     na=False
# )][["itemid", "label"]].head(20)

In [ ]:
temp_f_converted = (uniform["temp(F)"] - 32) * 5 / 9
uniform["temp(C)"] = uniform["temp(C)"].combine_first(temp_f_converted)

In [ ]:
vitals_new=uniform.columns
vitals_new

In [ ]:
(uniform[vitals_new].isna().mean() * 100).sort_values(ascending=False)

In [ ]:
uniform.sort_values(['stay_id','charttime'])
uniform.head(10)
#df.sort_values('a', ascending=False)

In [ ]:
import pandas as pd
import pyarrow as pa

print(pd.__version__)
print(pa.__version__)

In [ ]:
import pyarrow
uniform.to_parquet("uniform_20min.parquet")


Parquet is basically a data file format designed for big datasets. 
Think of it as a much smarter alternative to CSV.

In [ ]:
import os
os.path.getsize("uniform_20min.parquet") / 1024**3
#occupies much less space.

# THIS IS THE CHECKPOINT FOR RECOVERING THE UNIFORM DATAFRAME WHICH WAS SAVED AS A PARQUET. RUN THIS LINE FOR IMPLMENTATIONS HENCE FORTH

In [24]:
import os
import pandas as pd
import numpy as np
import duckdb

In [25]:
d_items_path='../data/d_items.csv.gz'
icustays_path='../data/icustays.csv.gz'
patients_path='../data/patients.csv.gz'
admissions_path='../data/admissions.csv.gz'
chartevents_path='../data/chartevents.csv.gz'
d_items=pd.read_csv(d_items_path)
icustays=pd.read_csv(icustays_path)
patients=pd.read_csv(patients_path)
admissions=pd.read_csv(admissions_path)
print("all the files have been converted to csvs")

all the files have been converted to csvs


In [26]:
uniform = pd.read_parquet("uniform_20min.parquet")

In [58]:
uniform.head()

,stay_id,ARTmean,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C),temp(F),window_id
charttime,,,,,,,,,,,
2174-09-29 12:00:00,30000153,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.0,96.8,0
2174-09-29 12:20:00,30000153,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.0,96.8,0
2174-09-29 12:40:00,30000153,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.0,96.8,0
2174-09-29 13:00:00,30000153,NaN,77.0,84.0,113.0,16.0,104.0,100.0,36.0,96.8,0
2174-09-29 13:20:00,30000153,NaN,77.0,84.0,113.0,16.0,104.0,100.0,36.0,96.8,0


In [ ]:
#optional
print(uniform.index.name)
print(uniform.columns)

In [ ]:
#optional
diffs = (
    uniform.groupby("stay_id", sort=False)
    .apply(lambda x: x.index.to_series().diff())
    .dropna()
)
#check the spacing thruout
diffs.value_counts().head()

charttime
0 days 00:20:00    24204470
Name: count, dtype: int64

In [29]:
uniform = uniform.sort_values(["stay_id", uniform.index.name])

In [30]:
#taking 2 hr windows now
window_size = 6
uniform["window_id"]=(uniform.groupby("stay_id")).cumcount()//window_size
uniform.head(10)


,stay_id,ARTmean,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C),temp(F),window_id
charttime,,,,,,,,,,,
2174-09-29 12:00:00,30000153,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.000000,96.8,0
2174-09-29 12:20:00,30000153,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.000000,96.8,0
2174-09-29 12:40:00,30000153,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.000000,96.8,0
2174-09-29 13:00:00,30000153,NaN,77.0,84.0,113.0,16.0,104.0,100.0,36.000000,96.8,0
2174-09-29 13:20:00,30000153,NaN,77.0,84.0,113.0,16.0,104.0,100.0,36.000000,96.8,0
2174-09-29 13:40:00,30000153,NaN,77.0,84.0,113.0,16.0,104.0,100.0,NaN,NaN,0
2174-09-29 14:00:00,30000153,NaN,77.0,84.0,113.0,16.0,104.0,100.0,37.277778,99.1,1
2174-09-29 14:20:00,30000153,NaN,77.0,84.0,113.0,16.0,104.0,100.0,37.277778,99.1,1
2174-09-29 14:40:00,30000153,NaN,NaN,NaN,NaN,16.0,83.0,100.0,37.277778,99.1,1


In [ ]:
#optional
uniform.columns

In [31]:
vitalsnew=[
    "ARTmean",
    "NBPd", "NBPm", "NBPs",
    "RR", "heartrate", "spO2", "temp(C)"
]

In [32]:
completeness = (
    uniform[vitalsnew].notna()
    .groupby([uniform["stay_id"], uniform["window_id"]])
    .mean()
)

In [ ]:
#optional
completeness.head()

In [ ]:
#optional
completeness.describe()

In [33]:
morethanthresrows=completeness[completeness[["RR","heartrate","spO2"]]>=0.8].all(axis=1)
morethanthresrows.head()
stayidsmorethanthres=morethanthresrows.index.get_level_values("stay_id").unique()
stayidsmorethanthres

Index([30000153, 30000213, 30000484, 30000646, 30000831, 30001148, 30001336,
       30001396, 30001446, 30001471,
       ...
       39999168, 39999172, 39999230, 39999286, 39999301, 39999384, 39999552,
       39999562, 39999810, 39999858],
      dtype='int64', name='stay_id', length=93729)

In [34]:
uniform_reset = uniform.reset_index() 
uniform_reset.head()

,charttime,stay_id,ARTmean,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C),temp(F),window_id
0,2174-09-29 12:00:00,30000153,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.0,96.8,0
1,2174-09-29 12:20:00,30000153,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.0,96.8,0
2,2174-09-29 12:40:00,30000153,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.0,96.8,0
3,2174-09-29 13:00:00,30000153,NaN,77.0,84.0,113.0,16.0,104.0,100.0,36.0,96.8,0
4,2174-09-29 13:20:00,30000153,NaN,77.0,84.0,113.0,16.0,104.0,100.0,36.0,96.8,0


In [35]:
 # charttime becomes a normal column now
uniform_filtered = uniform_reset.set_index(["stay_id","window_id"]).loc[morethanthresrows.index].reset_index()
uniform_filtered.head()

,stay_id,window_id,charttime,ARTmean,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C),temp(F)
0,30000153,0,2174-09-29 12:00:00,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.0,96.8
1,30000153,0,2174-09-29 12:20:00,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.0,96.8
2,30000153,0,2174-09-29 12:40:00,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.0,96.8
3,30000153,0,2174-09-29 13:00:00,NaN,77.0,84.0,113.0,16.0,104.0,100.0,36.0,96.8
4,30000153,0,2174-09-29 13:20:00,NaN,77.0,84.0,113.0,16.0,104.0,100.0,36.0,96.8


taking only the rows which pass that threshold, then removing that index and then setting backt he index

In [36]:
#optional
uniform_filtered.columns

Index(['stay_id', 'window_id', 'charttime', 'ARTmean', 'NBPd', 'NBPm', 'NBPs',
       'RR', 'heartrate', 'spO2', 'temp(C)', 'temp(F)'],
      dtype='str')

In [ ]:
#optional
uniform_filtered.head()

In [37]:
uniform_filtered.drop(columns=["temp(F)"])

,stay_id,window_id,charttime,ARTmean,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C)
0,30000153,0,2174-09-29 12:00:00,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.000000
1,30000153,0,2174-09-29 12:20:00,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.000000
2,30000153,0,2174-09-29 12:40:00,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.000000
3,30000153,0,2174-09-29 13:00:00,NaN,77.0,84.0,113.0,16.0,104.0,100.0,36.000000
4,30000153,0,2174-09-29 13:20:00,NaN,77.0,84.0,113.0,16.0,104.0,100.0,36.000000
...,...,...,...,...,...,...,...,...,...,...,...
24298194,39999858,44,2167-04-30 06:40:00,NaN,NaN,NaN,NaN,27.0,68.0,94.0,NaN
24298195,39999858,45,2167-04-30 07:00:00,NaN,NaN,NaN,NaN,26.0,96.0,88.0,NaN
24298196,39999858,45,2167-04-30 07:20:00,NaN,NaN,NaN,NaN,26.0,96.0,88.0,NaN
24298197,39999858,45,2167-04-30 07:40:00,NaN,NaN,NaN,NaN,26.0,96.0,88.0,NaN


In [38]:
uniform_filtered["window_uid"]=(uniform_filtered["stay_id"].astype(str)+"_"+uniform_filtered["window_id"].astype(str))
uniform_filtered.head()

,stay_id,window_id,charttime,ARTmean,NBPd,NBPm,NBPs,RR,heartrate,spO2,temp(C),temp(F),window_uid
0,30000153,0,2174-09-29 12:00:00,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.0,96.8,30000153_0
1,30000153,0,2174-09-29 12:20:00,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.0,96.8,30000153_0
2,30000153,0,2174-09-29 12:40:00,NaN,74.0,89.0,136.0,18.0,100.0,100.0,36.0,96.8,30000153_0
3,30000153,0,2174-09-29 13:00:00,NaN,77.0,84.0,113.0,16.0,104.0,100.0,36.0,96.8,30000153_0
4,30000153,0,2174-09-29 13:20:00,NaN,77.0,84.0,113.0,16.0,104.0,100.0,36.0,96.8,30000153_0


In [39]:
vitals1 = ["heartrate", "spO2", "RR", "NBPd", "NBPm", "NBPs", "ARTmean", "temp(C)"]

In [40]:
long_df=uniform_filtered.melt(id_vars=["window_uid","charttime"],value_vars=vitals1,var_name="vital",value_name="reading").dropna(subset=["reading"])
#df.melt(id_vars=['a'], value_vars=['b','c'])

In [41]:
long_df.to_parquet("long_df_full.parquet")
#please use the parquet henceforth

In [ ]:
long_df.head()

next is the tsfresh part - tsfresh (Time Series Feature extraction based on scalable hypothesis tests) is a library which extracts distinct features from time series data.

In [42]:
from tsfresh import extract_features
from tsfresh.feature_extraction import EfficientFCParameters
from tsfresh.utilities.dataframe_functions import impute

When we use EfficientFCParameters(), tsfresh runs about 70-90 math formulas on every window. Here are the three main types of "facts" it figures out:
Distribution Facts: What is the mean, median, max, min, and standard deviation? (Standard stuff).
Temporal Facts (Time-based):
Linear Trend: What is the slope of the line? (Is the patient getting worse or better?)
Absolute Energy: How much "power" is in the signal?
Complexity: Is the signal "noisy" and jumping around, or is it smooth?
Position Facts: At what point in the 2 hours did the maximum heart rate occur? Was it at the beginning or the end?
The result: It turns your 6 rows of "Raw Data" into one single row of "Mathematical Summary."

When you run `tsfresh` with `EfficientFCParameters()`, you are essentially asking the computer to describe your 2-hour ICU windows using math instead of raw numbers. 

Here is the breakdown of the "facts" it calculates for every vital sign in every window:

### 1. Distribution Facts (The "Spread")
These describe the general range and center of the data. 
*   **Mean & Median:** The average and middle values.
*   **Max & Min:** The extremes of the 2-hour window.
*   **Standard Deviation:** How much the vital sign "bounces" around the average.
*   **Skewness:** Is the data balanced, or are there "outlier" spikes in one direction?

### 2. Temporal Facts (The "Story" of the Signal)
These look at how the vital sign changes over time. This is where the **ECE (Signal Processing)** logic comes in.
*   **Linear Trend:** What is the slope of the line? (e.g., Is the heart rate steadily climbing or dropping?)
*   **Absolute Energy:** A measure of the total "power" or magnitude of the signal over those 2 hours.
*   **Complexity (Complexity Invariant Distance):** Is the signal "noisy" and jumping around erratically, or is it a smooth, predictable line?

### 3. Position Facts (The "Timing")
These tell you *when* things happened within the 2-hour window.
*   **Location of Max/Min:** Did the patient's heart rate peak at the very beginning of the window or at the very end? 
*   **Longest Strike:** Does the signal stay above or below its mean for a long time, or does it cross the average frequently?

---

### The Result: Dimensionality Reduction
The most important thing to understand is the **transformation**:

*   **Before `tsfresh`:** You have **6 rows** of raw measurements per window. A Machine Learning model can't easily "read" 6 rows at once to make a prediction.
*   **After `tsfresh`:** You have **1 single row** containing ~70 mathematical summaries. 

**It turns a "movie" (the 2-hour window) into a "still photo" (the feature summary) that a Machine Learning model can actually understand.**

---

**Wait, what happens next?**
Once you have this "still photo" (the `features_df`), you can feed it into an Anomaly Detection algorithm (like an Isolation Forest or an Autoencoder) to find the windows that look "weird" compared to the rest of the ICU stays.

**Ready to see how many windows you're about to process?** (Run `len(long_df)` to check the scale!)

In [43]:
settings = EfficientFCParameters()

In [44]:
all_uids = long_df['window_uid'].unique()
sample_uids = np.random.choice(all_uids, size=10000, replace=False)
long_df_sample = long_df[long_df['window_uid'].isin(sample_uids)]
#first get random 10k windows

So turns out feature extraction is superbly memeory heavy (obv) so the workflow is to split into 10 batches consisting of 10k each, save them as a parquet and this 100k set will be used to train the isolation forest

In [45]:

features_batch_1 = extract_features(
    long_df_sample, 
    column_id="window_uid", 
    column_sort="charttime", 
    column_kind="vital", 
    column_value="reading",
    default_fc_parameters=EfficientFCParameters(),
    n_jobs=4
)

features_batch_1.to_parquet("features_pilot_10k.parquet")


Feature Extraction: 100%|██████████| 20/20 [15:03<00:00, 45.20s/it]


In [ ]:
print(features_batch_1.shape)
print(features_batch_1.head())

In [46]:
features_cleaned = impute(features_batch_1)
#filling of missing values using 0 filling or any other statistical means since isolation forest doesn't work well with NaN values

/Users/medurarvind/ICU/venv/lib/python3.14/site-packages/tsfresh/utilities/dataframe_functions.py:198: RuntimeWarning: The columns <ArrowStringArray>
[                                        'NBPd__autocorrelation__lag_6',
                                         'NBPd__autocorrelation__lag_7',
                                         'NBPd__autocorrelation__lag_8',
                                         'NBPd__autocorrelation__lag_9',
                                 'NBPd__partial_autocorrelation__lag_3',
                                 'NBPd__partial_autocorrelation__lag_4',
                                 'NBPd__partial_autocorrelation__lag_5',
                                 'NBPd__partial_autocorrelation__lag_6',
                                 'NBPd__partial_autocorrelation__lag_7',
                                 'NBPd__partial_autocorrelation__lag_8',
 ...
  'ARTmean__agg_linear_trend__attr_"stderr"__chunk_len_10__f_agg_"min"',
 'ARTmean__agg_linear_trend__attr_"stderr"

In [47]:
features_cleaned = features_cleaned.loc[:, features_cleaned.nunique() > 1]
print(f"Final shape for ML: {features_cleaned.shape}")

Final shape for ML: (10000, 2043)


ISOLATION FOREST PART

In [48]:
from sklearn.ensemble import IsolationForest

In [49]:
isoforest=IsolationForest(contamination=0.05,random_state=42,n_jobs=-1)
features_cleaned["label"]=isoforest.fit_predict(features_cleaned) #pass or fail type
features_cleaned["score"]=isoforest.decision_function(features_cleaned.drop(columns=["label"])) #how anomalous it is

In [50]:
features_cleaned["score"].head()

30000646_12     0.040218
30001336_11     0.066914
30002012_3      0.096424
30004018_119   -0.002425
30004018_128    0.024949
Name: score, dtype: float64

In [51]:
worst_windows=features_cleaned["score"].nsmallest(5)
print(worst_windows)

34882006_2     -0.176237
38669449_9     -0.150766
36943279_68    -0.129960
39784741_296   -0.103929
34846900_18    -0.101589
Name: score, dtype: float64


In [54]:
# Pull the raw data for the #1 anomaly
top_id='34882006_2'
raw_vitals=uniform_filtered[uniform_filtered['window_uid']==top_id]
print(raw_vitals[['charttime', 'heartrate', 'spO2', 'RR', 'NBPs', 'NBPd', 'temp(C)']])

print(raw_vitals[['heartrate', 'spO2', 'RR', 'NBPs', 'NBPd', 'temp(C)']].agg(['mean', 'std', 'min', 'max']))

                   charttime  heartrate   spO2    RR   NBPs   NBPd  temp(C)
11962965 2169-05-21 05:20:00      138.0  100.0  11.0  143.0  109.0      NaN
11962966 2169-05-21 05:40:00      138.0  100.0  11.0  143.0  109.0      NaN
11962967 2169-05-21 06:00:00      124.0   65.0  34.0   63.0   39.0      NaN
11962968 2169-05-21 06:20:00      124.0   65.0  34.0   63.0   39.0      NaN
11962969 2169-05-21 06:40:00      124.0   65.0  34.0   63.0   39.0      NaN
11962970 2169-05-21 07:00:00      155.0   44.0   0.0  159.0   98.0      NaN
       heartrate        spO2         RR        NBPs        NBPd  temp(C)
mean  133.833333   73.166667  20.666667  105.666667   72.166667      NaN
std    12.432484   22.319648  15.148157   47.102725   36.553614      NaN
min   124.000000   44.000000   0.000000   63.000000   39.000000      NaN
max   155.000000  100.000000  34.000000  159.000000  109.000000      NaN


In [55]:

np.save("processed_uids_batch1.npy", sample_uids)

BATCH 2

In [56]:
import numpy as np
sample_uids = np.load("processed_uids_batch1.npy", allow_pickle=True)
#recover


In [57]:
remaining_uids = np.setdiff1d(all_uids, sample_uids)
sample_uids_2 = np.random.choice(remaining_uids, size=10000, replace=False)
long_df_sample_2 = long_df[long_df['window_uid'].isin(sample_uids_2)]

In [ ]:
print(long_df_sample_2.columns.tolist())

In [ ]:

from tsfresh.feature_extraction import EfficientFCParameters

fc_parameters = EfficientFCParameters()
fc_parameters.pop("binned_entropy", None)

features_batch_2 = extract_features(
    long_df_sample_2,  # whichever batch you're on
    column_id="window_uid",
    column_sort="charttime",
    column_kind="vital",      # remember: your actual column names are vital/reading, not kind/value
    column_value="reading",
    default_fc_parameters=fc_parameters,
    n_jobs=7
)
features_batch_2.to_parquet("features_pilot_10k_batch2.parquet")

In [ ]:
features_cleaned2 = impute(features_batch_2)

In [ ]:
features_cleaned2 = features_cleaned2.loc[:, features_cleaned2.nunique() > 1]
print(f"Final shape for ML: {features_cleaned2.shape}")

In [ ]:
isoforest=IsolationForest(contamination=0.05,random_state=42,n_jobs=-1)
features_cleaned2["label"]=isoforest.fit_predict(features_cleaned2) #pass or fail type
features_cleaned2["score"]=isoforest.decision_function(features_cleaned2.drop(columns=["label"])) #how anomalous it is

In [ ]:
features_cleaned2["score"].head()

In [ ]:
worst_windows=features_cleaned2["score"].nsmallest(5)
print(worst_windows)

In [ ]:
top_id='32603397_7'
raw_vitals2=uniform_filtered[uniform_filtered['window_uid']==top_id]
print(raw_vitals2[['charttime', 'heartrate', 'spO2', 'RR', 'NBPs', 'NBPd', 'temp(C)']])

print(raw_vitals2[['heartrate', 'spO2', 'RR', 'NBPs', 'NBPd', 'temp(C)']].agg(['mean', 'std', 'min', 'max']))

batch 3 of isolation forest

In [ ]:
used_uids = np.concatenate([sample_uids, sample_uids_2])  # append each new batch here

remaining = np.setdiff1d(all_uids, used_uids)
sample_uids_3 = np.random.choice(remaining, size=10000, replace=False)
long_df_sample_3 = long_df[long_df['window_uid'].isin(sample_uids_3)]
used_uids = np.concatenate([used_uids, sample_uids_3])  # update after each batch

#much cleaner method to exclude ids this way


In [ ]:
print(long_df_sample_2.columns.tolist())

syntax
features_batch_2 = extract_features(
    <whichever dataframe >,
    column_id="<the name of the column>",
    column_sort="<sorting function>",
    column_kind="<name of the other first>",
    column_value=<name of the corresponding value>,
    default_fc_parameters=fc_parameters (defauly),
    n_jobs=7
)
features_batch_2.to_parquet("features_pilot_10k_batch2.parquet")

In [ ]:
features_batch_3 = extract_features(
    long_df_sample_3,
    column_id="window_uid",
    column_sort="charttime",
    column_kind="vital",
    column_value="reading",
    default_fc_parameters=fc_parameters,
    n_jobs=7
    )
features_batch_3.to_parquet("features_pilot_10k_batch3.parquet")

In [ ]:
features_cleaned3 = impute(features_batch_3)

In [ ]:
features_cleaned3 = features_cleaned3.loc[:, features_cleaned3.nunique() > 1]

In [ ]:
isoforest=IsolationForest(contamination=0.05,random_state=42,n_jobs=-1)
features_cleaned3["label"]=isoforest.fit_predict(features_cleaned3) #pass or fail type
features_cleaned3["score"]=isoforest.decision_function(features_cleaned3.drop(columns=["label"])) #how anomalous it is

In [ ]:
features_cleaned3["score"].head()

In [ ]:
worst_windows=features_cleaned2["score"].nsmallest(5)
print(worst_windows)